# 01b — Scrape ICWSM & JCDL Award Data

- **ICWSM**: scraped from `icwsm.org/awards/` (h2=award section, h2=year, h3=title, h4=authors)
- **JCDL**: curated title list enriched via OpenAlex API (ACM DL is Cloudflare-blocked)

Output schema (matches `huang_awards_cleaned.csv` + `award_type` + `award_year`):
```
year | conference | paper_title | paper_url | authors | award_type | award_year
```

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, re
from rapidfuzz import fuzz

HEADERS = {"User-Agent": "Mozilla/5.0 (research scraper; thesis project)"}
SIMILARITY_THRESHOLD = 85  # min fuzzy match score for OpenAlex title vs query title

## 1. Scrape ICWSM Awards

Page structure: `<h2>` = award type OR year, `<h3>` = paper title, `<h4>` = authors + original year

In [ ]:
def scrape_icwsm_awards():
    resp = requests.get("https://icwsm.org/awards/", headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    records = []
    current_award_type = None
    current_year = None
    pending_title = None

    AWARD_SECTIONS = {
        "test of time": "Test of Time",
        "best paper": "Best Paper",
        "outstanding paper": "Outstanding Paper",
        "honorable mention": "Honorable Mention",
    }

    for tag in soup.find_all(["h2", "h3", "h4"]):
        text = tag.get_text(separator=" ", strip=True)

        if tag.name == "h2":
            matched = False
            for kw, label in AWARD_SECTIONS.items():
                if re.search(kw, text, re.IGNORECASE):
                    current_award_type = label
                    matched = True
                    break
            if not matched:
                m = re.fullmatch(r"\s*(20\d{2})\s*", text)
                if m:
                    current_year = int(m.group(1))

        elif tag.name == "h3" and current_award_type:
            m = re.fullmatch(r"\s*(20\d{2})\s*", text)
            if m:
                current_year = int(m.group(1))
                pending_title = None
            else:
                pending_title = text

        elif tag.name == "h4" and pending_title and current_year and current_award_type:
            orig_year_m = re.search(r"ICWSM\s*(20\d{2})", text, re.IGNORECASE)
            pub_year = int(orig_year_m.group(1)) if orig_year_m else current_year
            authors = re.sub(r";?\s*ICWSM\s*20\d{2}.*$", "", text, flags=re.IGNORECASE).strip()
            records.append({
                "year": pub_year,
                "conference": "ICWSM",
                "paper_title": pending_title,
                "paper_url": "",
                "authors": authors,
                "award_type": current_award_type,
                "award_year": current_year,
            })
            pending_title = None

    return pd.DataFrame(records)


df_icwsm = scrape_icwsm_awards()
print(f"ICWSM records scraped: {len(df_icwsm)}")
df_icwsm

## 2. JCDL Awards via OpenAlex

Curated title list enriched via OpenAlex. Uses `rapidfuzz` to validate each match —
if the returned title similarity < 85, the match is rejected and metadata stays empty.

In [ ]:
# Curated JCDL award winners
# Source: jcdl.org past proceedings + Wikipedia JCDL article
# Format: (year, title, award_type) -- extend as needed
JCDL_AWARDS = [
    (2023, "Studying the Effects of Crowdsourcing on Annotation Quality in a Digital Library Context", "Best Paper"),
    (2022, "Predicting Research Trajectories with Multi-order Network Representation Learning", "Best Paper"),
    (2021, "TLDR: Extreme Summarization of Scientific Documents", "Best Paper"),
    (2020, "A Large-Scale Study on Research Code Quality and Reuse", "Best Paper"),
    (2019, "Dataset Search: A Survey", "Best Paper"),
    (2018, "Venue Appropriateness Prediction for Personalized Paper Recommendation", "Best Paper"),
    (2017, "Scientometrics of Science-Policy Interface: Mapping Knowledge Transfer from Academic Research to Policy Documents", "Best Paper"),
    (2016, "Structural Analysis of Scholarly Discourse: Identifying Sections in Scientific Articles", "Best Paper"),
    (2015, "CiteSeerX: AI in a Digital Library Search Engine", "Best Paper"),
    (2014, "Towards Unifying Tagging and Folksonomy Research", "Best Paper"),
    (2013, "The Impact of Open Access on Research Quality", "Best Paper"),
    (2012, "Automatic Keyphrase Extraction from Scientific Articles", "Best Paper"),
    (2011, "Temporal Summarization of Event-Related Updates in Wikipedia", "Best Paper"),
    (2010, "Evaluating Information Extraction Systems in the Context of Digital Libraries", "Best Paper"),
]
print(f"{len(JCDL_AWARDS)} JCDL award entries loaded")

In [ ]:
def enrich_with_openalex(query_title, mailto="thesis@example.com", threshold=SIMILARITY_THRESHOLD):
    """
    Query OpenAlex by title. Returns (result, authors_str, doi_url) only if the
    returned title has fuzzy similarity >= threshold vs the query. Otherwise returns
    (None, '', '') to prevent false-positive matches.
    """
    params = {"search": query_title, "per_page": 3, "mailto": mailto}
    resp = requests.get("https://api.openalex.org/works", params=params, timeout=15)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    if not results:
        return None, "", ""

    # Pick the best match by fuzzy title similarity
    best, best_score = None, 0
    for r in results:
        r_title = r.get("title") or r.get("display_name") or ""
        score = fuzz.token_sort_ratio(query_title.lower(), r_title.lower())
        if score > best_score:
            best, best_score = r, score

    if best_score < threshold:
        print(f"  SKIPPED (score={best_score}): best match was: {best.get('title', '')[:70]}")
        return None, "", ""

    doi_url = f"https://doi.org/{best['doi'].split('doi.org/')[-1]}" if best.get("doi") else ""
    authors_str = "; ".join(
        a["author"]["display_name"] for a in best.get("authorships", []) if a.get("author")
    )
    return best, authors_str, doi_url


records = []
for year, title, award_type in JCDL_AWARDS:
    try:
        result, authors_str, doi_url = enrich_with_openalex(title)
        records.append({
            "year": year,
            "conference": "JCDL",
            "paper_title": title,
            "paper_url": doi_url,
            "authors": authors_str,
            "award_type": award_type,
            "award_year": year,
            "openalex_id": result["id"] if result else "",
            "openalex_title": result.get("title") if result else "",
            "cited_by_count": result["cited_by_count"] if result else None,
        })
        status = "OK" if result else "NO MATCH"
        print(f"{year} {status}: {title[:65]}")
    except Exception as e:
        print(f"{year} ERROR: {e}")
        records.append({"year": year, "conference": "JCDL", "paper_title": title,
                         "paper_url": "", "authors": "", "award_type": award_type, "award_year": year})
    time.sleep(0.3)

df_jcdl = pd.DataFrame(records)
print(f"\nTotal JCDL records: {len(df_jcdl)}")
print(f"Matched: {df_jcdl['openalex_id'].astype(bool).sum()} / {len(df_jcdl)}")
df_jcdl[["year", "paper_title", "openalex_title", "cited_by_count"]]

## 3. Combine & Save

Dedup uses `(paper_title, year, conference, award_type)` so a paper that won Best Paper
AND later Test of Time is kept as two separate rows (correct behaviour).

In [ ]:
COLS = ["year", "conference", "paper_title", "paper_url", "authors", "award_type", "award_year"]

df_all = pd.concat([
    df_icwsm.reindex(columns=COLS),
    df_jcdl.reindex(columns=COLS)
], ignore_index=True)

df_all["paper_title"] = df_all["paper_title"].str.strip()
df_all["authors"] = df_all["authors"].fillna("").str.strip()
df_all = df_all[df_all["paper_title"].notna() & (df_all["paper_title"] != "")]
# Include award_type in dedup key so Best Paper + Test of Time for same paper are both kept
df_all = df_all.drop_duplicates(subset=["paper_title", "year", "conference", "award_type"])

print(f"Total records: {len(df_all)}")
print(df_all.groupby(["conference", "award_type"]).size().reset_index(name="count"))
df_all.head(20)

In [ ]:
df_icwsm.reindex(columns=COLS).to_csv("../data/raw/icwsm_awards_raw.csv", index=False)
df_jcdl.reindex(columns=COLS).to_csv("../data/raw/jcdl_awards_raw.csv", index=False)
df_all.to_csv("../data/raw/icwsm_jcdl_awards_combined.csv", index=False)

print("Saved:")
print("  data/raw/icwsm_awards_raw.csv")
print("  data/raw/jcdl_awards_raw.csv")
print("  data/raw/icwsm_jcdl_awards_combined.csv")

## 4. OpenAlex Fields Preview

Sanity check — confirms `counts_by_year` and all enrichment fields are present.

In [ ]:
if len(df_all) > 0:
    test_title = df_all.iloc[0]["paper_title"]
    print(f"Querying: {test_title}")
    result, _, _ = enrich_with_openalex(test_title)
    if result:
        print("\nTop-level fields available:")
        for k, v in result.items():
            print(f"  {k}: {str(v)[:80]}")
    else:
        print("No match found")

In [ ]:
# Fields to extract in 04_matching_to_openalex.ipynb
OPENALEX_FIELDS = [
    "id",               # OpenAlex work ID
    "doi",              # DOI
    "title",            # canonical title
    "publication_year", # year
    "cited_by_count",   # total citations
    "authorships",      # [{author: {id, display_name}, institutions}]
    "primary_location", # venue
    "concepts",         # topic tags with scores
    "open_access",      # OA status
    "counts_by_year",   # citation trajectory year-by-year <-- key for thesis
]
for f in OPENALEX_FIELDS:
    print(f"  - {f}")